<a href="https://colab.research.google.com/github/Rixdorfer-Kindl/protein-design-pipline-colab/blob/main/RFdiffusion_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RFdiffusion on Colab

> **Version 2.5.0.** The current version is always the one at
> [the releases page](https://github.com/a-LittleMonk/RFdiffusion-GoogleColab/releases).
> If that page shows a higher number than the line you are reading, this copy
> is out of date — [open the current notebook](https://colab.research.google.com/github/a-LittleMonk/RFdiffusion-GoogleColab/blob/main/RFdiffusion_colab.ipynb) and *File > Save a
> copy in Drive* to replace it. A saved copy never updates itself.

Protein backbone generation. Run steps 1-6 once (~12 min), then use step 7 for designs.

**First: Runtime -> Change runtime type -> T4 GPU.**

<small>**Using Run all?** Two things need you: step 2 waits for a Google Drive
authorisation popup, and step 7's fields must already say what you want built - its
defaults make a 100-residue monomer and ignore any file in Drive. Set those two up and
the rest, including saving back to Drive, runs unattended. The setup steps now raise on
a bad environment, so a Run all stops at the real cause instead of cascading.</small>

<small>Runs RFdiffusion inside its own Python 3.10 environment, called as a program rather
than imported, because Colab's own Python is 3.13 and RFdiffusion's dependencies have no
3.13 builds. Double-click a step title to see its code.</small>

In [1]:
#@title 1. Check the GPU
#@markdown Must show a GPU. If not, fix the runtime type before going further.

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
Python 3.13.15


In [2]:
#@title 2. Mount your Google Drive
#@markdown Gives the notebook a folder on your Drive, so you can feed it a target
#@markdown structure you keep there and have the designs land back there too.
#@markdown A browser popup will ask you to authorise access.
drive_folder = "RFdiffusion" #@param {type:"string"}

import os
from datetime import datetime
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT   = f"/content/drive/MyDrive/{drive_folder}"
DRIVE_INPUTS = f"{DRIVE_ROOT}/inputs"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/outputs"
os.makedirs(DRIVE_INPUTS, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print(f"Drive mounted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Put target structures in:  My Drive/{drive_folder}/inputs")
print(f"Designs will be saved to:  My Drive/{drive_folder}/outputs\n")

found = sorted(f for f in os.listdir(DRIVE_INPUTS)
               if f.lower().endswith((".pdb", ".cif", ".ent")))
if found:
    print("Structures already in your inputs folder - paste one of these names")
    print("into `input_pdb` in step 7:")
    for f in found:
        print("   ", f)
else:
    print("inputs folder is empty for now. Drop a .pdb there (the Files pane on the")
    print("left refreshes with the folder icon) and re-run this cell to list it.")


Mounted at /content/drive
Drive mounted at: 2026-10-21 14:12:59
Put target structures in:  My Drive/RFdiffusion/inputs
Designs will be saved to:  My Drive/RFdiffusion/outputs

inputs folder is empty for now. Drop a .pdb there (the Files pane on the
left refreshes with the folder icon) and re-run this cell to list it.


In [ ]:
#@title 3. Install Python 3.10
#@markdown Sits alongside Colab's own Python without disturbing it.

!apt-get update -qq
!apt-get install -y -qq python3.10 python3.10-venv python3.10-dev
!python3.10 --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
#@title 4. Build the environment (~6 min, ~3 GB)
#@markdown Installs PyTorch 2.0.1 / CUDA 11.8, DGL 1.1.3, the CUDA 11 runtime libraries
#@markdown DGL loads by hand, and NumPy 1.x. Let it finish - do not interrupt it or
#@markdown re-run it part way through.
#@markdown
#@markdown Ends by checking itself and **raising** if anything is wrong, so a Run all
#@markdown stops here rather than failing later for a misleading reason.

import os, glob

ENV = "/content/rfenv"
PY  = f"{ENV}/bin/python"
PIP = f"{ENV}/bin/pip"
SP  = f"{ENV}/lib/python3.10/site-packages"

# A half-built environment is worse than none: rebuild rather than install on top.
if not os.path.isfile(PY):
    print("creating environment...")
    os.system(f"rm -rf {ENV}")
    os.system(f"python3.10 -m venv {ENV}")
assert os.path.isfile(PY), "venv creation failed - check step 3's output"

!{PIP} install -q --upgrade pip setuptools wheel
!{PIP} install -q torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118

# DGL opens the CUDA 11 libraries itself through the system loader, so they must
# exist as real files regardless of what PyTorch bundles privately.
!{PIP} install -q nvidia-cublas-cu11 nvidia-cusparse-cu11 nvidia-cusolver-cu11 nvidia-cufft-cu11 nvidia-curand-cu11 nvidia-cuda-runtime-cu11 nvidia-cuda-nvrtc-cu11 nvidia-cudnn-cu11
!{PIP} install -q dgl==1.1.3 -f https://data.dgl.ai/wheels/cu118/repo.html
!{PIP} install -q hydra-core omegaconf icecream e3nn==0.5.1 opt_einsum opt_einsum_fx biopython decorator pyrsistent pynvml jedi tqdm

# Pinned last so nothing drags NumPy 2 back in. PyTorch 2.0.1 was compiled against
# NumPy 1.x and fails to initialise against 2.x.
!{PIP} install -q "numpy==1.26.4" "scipy==1.11.4" "pandas==2.0.3"

# 修复关键：除了指向虚拟环境内 nvidia 的 CUDA 库外，额外将宿主系统的 CUDA /兼容库路径加入 LD_LIBRARY_PATH
# 这样可以确保虚拟环境内的 PyTorch 2.0.1 能够顺利通过 libcuda.so 与宿主机的 Tesla T4 GPU 驱动通信
SYSTEM_CUDA_PATHS = ["/usr/lib64-nvidia", "/usr/local/cuda/lib64"]
LDPATH = ":".join(sorted(glob.glob(f"{SP}/nvidia/*/lib")) + [f"{SP}/torch/lib"] + SYSTEM_CUDA_PATHS)

print("\n--- environment check ---")
hit = glob.glob(f"{SP}/**/libcusparse.so.11", recursive=True)
print("libcusparse.so.11:", hit[0] if hit else "*** NOT FOUND ***")

lines = [
    "import sys, torch, dgl, numpy",
    "print('python:', sys.version.split()[0])",
    "print('torch :', torch.__version__)",
    "print('dgl   :', dgl.__version__)",
    "print('numpy :', numpy.__version__)",
    "print('cuda  :', torch.cuda.is_available())",
    "print('gpu   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')",
    "print('torch<->numpy:', torch.tensor([1.0]).numpy().sum())",
]
with open("/content/_check.py", "w") as f:
    f.write("\n".join(lines))

import subprocess
_chk = subprocess.run([PY, "/content/_check.py"],
                      env=dict(os.environ, DGLBACKEND="pytorch", LD_LIBRARY_PATH=LDPATH),
                      capture_output=True, text=True)
print(_chk.stdout + _chk.stderr)

# Verified rather than left for the eye, so an unattended Run all halts on the real
# cause instead of carrying a broken environment into the steps below.
_vals = dict(l.split(":", 1) for l in _chk.stdout.splitlines() if ":" in l)
_vals = {k.strip(): v.strip() for k, v in _vals.items()}
_bad = []
if _chk.returncode:
    _bad.append("the check script itself failed - see its output above")
if not hit:
    _bad.append("libcusparse.so.11 is missing: the nvidia-*-cu11 installs failed, "
                "usually from a full disk")
if not _vals.get("numpy", "").startswith("1."):
    _bad.append(f"numpy is {_vals.get('numpy', '?')}, must be 1.x - something "
                "reinstalled NumPy 2 over the pin")
if _vals.get("cuda") != "True":
    _bad.append("torch cannot see the GPU - check the runtime type (step 1)")
if not _vals.get("dgl", "").startswith("1.1"):
    _bad.append(f"dgl is {_vals.get('dgl', 'missing or failed to import')}, must be 1.1.x")

if _bad:
    print("\n*** ENVIRONMENT IS NOT USABLE ***")
    for _b in _bad:
        print("  - " + _b)
    raise RuntimeError(
        "Environment check failed (" + str(len(_bad)) + " problem(s), listed above). "
        "Do NOT re-run this cell on top of the failure: Runtime > Disconnect and "
        "delete runtime, then start again from step 1.")

print("\nEnvironment OK.")

creating environment...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.3/876.3 kB 63.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 21.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

--- environment check ---
libcusparse.so.11: /content/rfenv/lib/python3.10/site-packages/nvidia/cusparse/lib/libcusparse.so.11
python: 3.10.22
torch : 2.0.1+cu118
dgl   : 1.1.3+cu118
numpy : 1.26.4
cuda  : True
gpu   : Tesla T4
torch<->numpy: 1.0


Environment OK.


**Check step 4's output before continuing.** You want `numpy: 1.26.4`, `dgl: 1.1.3`,
`cuda: True`, a GPU name, and a `torch<->numpy` line printing `1.0`. If DGL raised an
error or NumPy reads 2.x, stop here - nothing below will work, and the errors it produces
will point somewhere misleading.

In [4]:
#@title 5. Download RFdiffusion and model weights (~1 GB)
#@markdown Fetches the two checkpoints RFdiffusion needs, then keeps a copy in
#@markdown `My Drive/<your folder>/weights`. Later sessions reuse that copy
#@markdown instead of downloading again, which also means an outage on the
#@markdown download server cannot strand you. Run step 2 first to get it.
weights_mirror = "" #@param {type:"string"}
#@markdown <small>**Leave blank.** Only set this if the official server is down
#@markdown *and* you have a source you trust. Give a folder URL; the two file
#@markdown names are appended to it. The official download publishes no
#@markdown checksum - the hash in its URL is the MD5 of the words "hello world",
#@markdown not of the file - so a mirror cannot be verified against the real
#@markdown thing, and a PyTorch checkpoint runs code when it is loaded. That is
#@markdown why this is not filled in for you.</small>

import os, shutil, socket, time, urllib.request
%cd /content

if not os.path.isdir("/content/RFdiffusion"):
    !git clone -q https://github.com/sokrypton/RFdiffusion.git

MODELS = "/content/RFdiffusion/models"
os.makedirs(MODELS, exist_ok=True)

# No schedules download here. That file (files.ipd.uw.edu/krypton/schedules.zip)
# was removed from the server and now returns 404. It only ever held a cache:
# RFdiffusion computes the IGSO3 schedule itself when the folder is absent and
# saves it, so the first design run takes a few extra minutes and later ones do not.

OFFICIAL = "http://files.ipd.uw.edu/pub/RFdiffusion"
WEIGHTS = {
    "Base_ckpt.pt": OFFICIAL + "/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt",
    "Complex_base_ckpt.pt": OFFICIAL + "/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt",
}
# A real checkpoint is around 460 MB. Anything much smaller is a truncated
# download or an HTML error page saved under a .pt name.
MIN_BYTES = 100_000_000

DRIVE_WEIGHTS = (os.path.join(DRIVE_ROOT, "weights")
                 if "DRIVE_ROOT" in globals() else None)


def _usable(path):
    return os.path.isfile(path) and os.path.getsize(path) >= MIN_BYTES


def _bar(done, total, t0, note=""):
    secs = time.time() - t0
    rate = done / secs if secs > 0.5 else 0
    mb = done / 2 ** 20
    if total:
        pct = min(done * 100 / total, 100)
        eta = (total - done) / rate / 60 if rate else 0
        print(f"\r  [{('=' * int(pct // 2.5)).ljust(40)}] {pct:5.1f}%  "
              f"{mb:6.0f}/{total / 2 ** 20:.0f} MB  {rate / 2 ** 20:4.1f} MB/s  "
              f"ETA {eta:4.1f} min{note}", end="", flush=True)
    else:
        print(f"\r  {mb:6.0f} MB  {rate / 2 ** 20:4.1f} MB/s{note}",
              end="", flush=True)


def _fetch(url, dest):
    """Download with a progress bar, to a .part file so a failure is obvious."""
    part = dest + ".part"
    with urllib.request.urlopen(url, timeout=120) as r:
        total = int(r.headers.get("Content-Length") or 0)
        done, t0, painted = 0, time.time(), 0.0
        with open(part, "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
                done += len(chunk)
                if time.time() - painted > 1 or done == total:
                    painted = time.time()
                    _bar(done, total, t0)
        print()
    if os.path.getsize(part) < MIN_BYTES:
        os.remove(part)
        raise RuntimeError("the download was far too small to be a checkpoint")
    os.replace(part, dest)


def _reachable(host, port=80, timeout=8):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


_missing = []
for _name, _url in WEIGHTS.items():
    _local = os.path.join(MODELS, _name)

    if _usable(_local):
        print(f"{_name}: already here")
        continue

    # 1. a copy kept on Drive from an earlier session - no download at all
    if DRIVE_WEIGHTS:
        _cached = os.path.join(DRIVE_WEIGHTS, _name)
        if _usable(_cached):
            print(f"{_name}: copying from your Drive cache")
            shutil.copyfile(_cached, _local)
            continue

    # 2. otherwise download it
    _src = (weights_mirror.rstrip("/") + "/" + _name) if weights_mirror.strip() else _url
    if weights_mirror.strip():
        print(f"{_name}: downloading from the mirror you supplied")
        print("  this cannot be checked against the official file - no checksum exists")
    else:
        print(f"{_name}: downloading from files.ipd.uw.edu")

    try:
        if not weights_mirror.strip() and not _reachable("files.ipd.uw.edu"):
            raise RuntimeError("files.ipd.uw.edu is not answering")
        _fetch(_src, _local)
    except Exception as _e:
        print(f"  failed: {_e}")
        _missing.append(_name)
        continue

    # 3. keep it on Drive so this is the last time
    if DRIVE_WEIGHTS:
        os.makedirs(DRIVE_WEIGHTS, exist_ok=True)
        shutil.copyfile(_local, os.path.join(DRIVE_WEIGHTS, _name))
        print(f"  saved to your Drive cache, so it will not download again")

if _missing:
    raise RuntimeError(
        "Could not get: " + ", ".join(_missing) + ".\n"
        "The weights live on files.ipd.uw.edu, which is the Institute for "
        "Protein Design's file server, and it does go down - when it does there "
        "is nothing to fall back on, because the official download publishes no "
        "checksum and so no mirror can be verified against it.\n"
        "Options: wait and re-run this step; or, if you have the two .pt files "
        "already, put them in My Drive/<your folder>/weights and re-run, which "
        "skips the download entirely.")

if not DRIVE_WEIGHTS:
    print("\nDrive is not mounted, so nothing was cached. Run step 2 and re-run")
    print("this step to keep a copy and never download these again.")

!ls -lh /content/RFdiffusion/models/


/content
Base_ckpt.pt: downloading from files.ipd.uw.edu
  [========================================] 100.0%     461/461 MB  11.0 MB/s  ETA  0.0 min
Complex_base_ckpt.pt: downloading from files.ipd.uw.edu
  [========================================] 100.0%     461/461 MB   9.0 MB/s  ETA  0.0 min

Drive is not mounted, so nothing was cached. Run step 2 and re-run
this step to keep a copy and never download these again.
total 923M
-rw-r--r-- 1 root root 462M Oct 21 12:51 Base_ckpt.pt
-rw-r--r-- 1 root root 462M Oct 21 12:52 Complex_base_ckpt.pt


In [5]:
#@title 6. Compile SE3Transformer (~3 min)
#@markdown RFdiffusion's equivariant attention layers. Ends by testing the exact import
#@markdown chain that fails in the standard notebook. If it prints OK, setup is done.
#@markdown
#@markdown RFdiffusion itself is deliberately *not* pip-installed: this fork ships no
#@markdown setup.py at its root, so `pip install -e .` fails. Its modules are found by
#@markdown running from inside its own directory instead.

import os, glob
ENV = "/content/rfenv"; PY = f"{ENV}/bin/python"; PIP = f"{ENV}/bin/pip"
SP  = f"{ENV}/lib/python3.10/site-packages"
LDPATH = ":".join(sorted(glob.glob(f"{SP}/nvidia/*/lib")) + [f"{SP}/torch/lib"])

!cd /content/RFdiffusion/env/SE3Transformer && {PIP} install -q --no-deps .

import subprocess
_imp = subprocess.run([PY, "-c", "import inference.utils"], cwd="/content/RFdiffusion",
                      env=dict(os.environ, DGLBACKEND="pytorch", LD_LIBRARY_PATH=LDPATH),
                      capture_output=True, text=True)
print(_imp.stdout + _imp.stderr)

# A marker file, so the design step can tell setup finished rather than assuming it
_MARK = "/content/.rf_setup_ok"
if _imp.returncode:
    if os.path.isfile(_MARK):
        os.remove(_MARK)
    raise RuntimeError(
        "RFdiffusion still does not import - see the traceback above. This is the "
        "failure the whole setup exists to avoid, so nothing below will work. "
        "Runtime > Disconnect and delete runtime, then start again from step 1.")

for _w in ("Base_ckpt.pt", "Complex_base_ckpt.pt"):
    _p = f"/content/RFdiffusion/models/{_w}"
    if not os.path.isfile(_p) or os.path.getsize(_p) < 10_000_000:
        raise RuntimeError(f"{_w} is missing or truncated - re-run step 5.")

open(_MARK, "w").write("ok")
print("RFdiffusion imports: OK - setup complete")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

RFdiffusion imports: OK - setup complete


In [1]:
#@title 7. Generate designs
#@markdown ### Run mode
run_mode = "Production" #@param ["Production", "Export check - ~1 min, does a file come out", "Fast debug - your settings, 5 steps", "Smoke test - ignore fields, tiny monomer"]
#@markdown <small>`Production` runs exactly what the fields below say.
#@markdown
#@markdown `Export check` is the one for "will I get a file at the end" without waiting
#@markdown for a real run. It keeps your target and the *shape* of your contig but
#@markdown shrinks every segment to ~12 residues and diffuses 2 steps, then checks the
#@markdown PDB exists, reads it back, and copies it to Drive - the whole chain, in about
#@markdown a minute. Hotspots are dropped, because shrinking the target moves them out
#@markdown of range.
#@markdown
#@markdown `Fast debug` keeps your settings at full size and only cuts the sampling to 5
#@markdown steps and 1 design: slower than the export check, but it validates the real
#@markdown contig string and hotspots. `Smoke test` ignores every field and builds a
#@markdown 20-residue monomer, to prove the install itself works.
#@markdown
#@markdown None of the three produce usable designs - 2-5 diffusion steps cannot fold
#@markdown anything.</small>
#@markdown ---
#@markdown ### What to build
name = "test1" #@param {type:"string"}
design_type = "monomer" #@param ["monomer", "binder", "motif scaffolding", "partial diffusion"]
contigs = "A310-480/0 50-100" #@param {type:"string"}
#@markdown ### Target structure (binder / motif / partial only)
#@markdown Three ways to give one, in the order they are tried: the **name of a file in
#@markdown your Drive inputs folder** (`tgfb1_dimer.pdb`), a **full path**, or a
#@markdown **4-letter PDB code** such as `1D8R`, which is downloaded for you.
#@markdown Leave blank for a monomer.
input_pdb = "1ivo" #@param {type:"string"}
#@markdown Hotspot residues the binder should engage, e.g. `A59,A83,A91`. Strongly
#@markdown recommended for binder design - without them the binder lands anywhere.
hotspots = "A364,A367" #@param {type:"string"}
#@markdown ### Sampling
iterations = 25 #@param [25, 50, 100, 150, 200] {type:"raw"}
num_designs = 8 #@param [1, 2, 4, 8, 16, 32] {type:"raw"}

#@markdown ---
#@markdown **Contig syntax**
#@markdown
#@markdown | Goal | `contigs` | `input_pdb` |
#@markdown |---|---|---|
#@markdown | Monomer, fixed length 100 | `100-100` | blank |
#@markdown | Monomer, sampled 70-100 | `70-100` | blank |
#@markdown | Binder to chain A | `A1-150/0 70-100` | target |
#@markdown | Scaffold a motif | `40/A163-181/40` | target |
#@markdown | Keep chain A, rebuild rest | `A` | target |
#@markdown
#@markdown `/0` marks a chain break. Start with 50 iterations and 1 design to confirm the
#@markdown pipeline runs before committing time to a real batch.

import subprocess, os, glob, re
ENV = "/content/rfenv"; PY = f"{ENV}/bin/python"
if not os.path.isfile("/content/.rf_setup_ok"):
    raise RuntimeError(
        "Setup has not completed in this runtime. Run steps 3-6 first; step 6 writes "
        "the marker this checks. If they did run, one of them failed - scroll up.")
SP  = f"{ENV}/lib/python3.10/site-packages"
LDPATH = ":".join(sorted(glob.glob(f"{SP}/nvidia/*/lib")) + [f"{SP}/torch/lib"])
os.makedirs("/content/outputs", exist_ok=True)

# --- run mode -------------------------------------------------------------
# Three ways to cut a run down, because they answer different questions:
# Export check shrinks the *size* (the dominant cost - attention is quadratic in
# residue count), Fast debug shrinks only the sampling so the real contig and
# hotspots are still validated, Smoke test discards the settings entirely.
import re, time

CHECK = run_mode.startswith("Export")
FAST  = run_mode.startswith("Fast")
SMOKE = run_mode.startswith("Smoke")
DEBUG = CHECK or FAST or SMOKE

def _shrink_contigs(c, keep=12, design=12):
    """Cut every segment down while keeping the contig's structure intact.

    Target slices (A1-112) keep their start and take the first `keep` residues,
    so the numbering stays valid against the real PDB. Designed ranges collapse
    to a fixed small length. Chain breaks (/0) and bare chain ids (A) are left
    alone - a bare id means "the whole chain" and cannot be shrunk without
    guessing where it ends.
    """
    c = re.sub(r"([A-Za-z])(\d+)-(\d+)",
               lambda m: f"{m.group(1)}{m.group(2)}-"
                         f"{min(int(m.group(3)), int(m.group(2)) + keep - 1)}", c)
    c = re.sub(r"(?<![A-Za-z\d])(\d+)-(\d+)", f"{design}-{design}", c)
    c = re.sub(r"(?<![A-Za-z\d-])(\d+)(?![-\d])",
               lambda m: m.group(1) if m.group(1) == "0" else str(design), c)
    return c

if CHECK:
    _real_contigs, _real_name = contigs, name
    contigs = _shrink_contigs(contigs)
    name = f"{name}_exportcheck"
    iterations, num_designs = 15, 1  # 修正：避开 T < 15 限制
    _dropped_hotspots, hotspots = hotspots.strip(), ""
elif FAST:
    iterations, num_designs = 15, 1  # 修正：避开 T < 15 限制
elif SMOKE:
    name, design_type, contigs = "smoketest", "monomer", "20-20"
    input_pdb, hotspots = "", ""
    iterations, num_designs = 15, 1  # 修正：避开 T < 15 限制

if DEBUG:
    print("*" * 70)
    print(f"* {run_mode}")
    if CHECK:
        print(f"* contigs shrunk: {_real_contigs}  ->  {contigs}")
        if re.search(r"(?<![A-Za-z\d])[A-Za-z](?![\w-])", contigs):
            print("*   a bare chain id means the whole chain, so it was left at full")
            print("*   size - this check will be slower than ~1 min")
        if _dropped_hotspots:
            print(f"* hotspots dropped ({_dropped_hotspots}): the shrunk target no")
            print("*   longer contains those residues")
        print(f"* writing to {name}_* so your real outputs are untouched")
    print("* Running verification: 15 steps are used to bypass the T >= 15 limit.")
    print("*" * 70, "\n")

_t0 = time.time()

# Resolve the target: a name in the Drive inputs folder, a path, or a PDB code
DRIVE_INPUTS = globals().get("DRIVE_INPUTS", "/content/drive/MyDrive/RFdiffusion/inputs")

pdb_path = input_pdb.strip()
if pdb_path and not os.path.isfile(pdb_path):
    from_drive = os.path.join(DRIVE_INPUTS, pdb_path)
    if os.path.isfile(from_drive):
        pdb_path = from_drive
        print(f"using target from Drive: {pdb_path}")
    elif len(pdb_path) == 4:
        dest = f"/content/{pdb_path}.pdb"
        if not os.path.isfile(dest):
            print(f"fetching {pdb_path} from the PDB...")
            os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_path}.pdb -O {dest}")
        pdb_path = dest
    else:
        raise FileNotFoundError(
            f"'{input_pdb}' is not a file, not in {DRIVE_INPUTS}, and not a 4-letter "
            "PDB code. Run step 2 to mount Drive, check the spelling, or give a full path."
        )

if design_type != "monomer" and not pdb_path:
    raise ValueError(f"design_type is '{design_type}' but input_pdb is empty")

# --- target check ---------------------------------------------------------
AA3TO1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q",
    "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K",
    "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
    "TYR": "Y", "VAL": "V", "MSE": "M", "SEC": "U", "PYL": "O",
}

chain_res, problems, bar = {}, [], "=" * 68

if pdb_path:
    with open(pdb_path) as _fh:
        pdb_lines = _fh.readlines()
    atoms = [l for l in pdb_lines if l.startswith("ATOM")]
    if not atoms:
        raise ValueError(f"{pdb_path} has no ATOM records - is it really a PDB?")
    seen = set()
    for l in atoms:
        ch, rid, rn = l[21], l[22:27].strip(), l[17:20].strip()
        if (ch, rid) not in seen:
            seen.add((ch, rid))
            chain_res.setdefault(ch, []).append((rid, rn))

    def resnum(rid):
        digits = "".join(c for c in rid if c.isdigit() or c == "-")
        return int(digits) if digits else None

    nums = {c: {resnum(r) for r, _ in v if resnum(r) is not None}
            for c, v in chain_res.items()}

    contig_chains = set(re.findall(r"([A-Za-z])\d", contigs))
    contig_chains |= set(re.findall(r"(?<![A-Za-z\d])([A-Za-z])(?![\w-])", contigs))

    for ch, lo, hi in re.findall(r"([A-Za-z])(\d+)-(\d+)", contigs):
        if ch not in chain_res:
            problems.append(f"contig wants chain {ch}, which is not in the file "
                            f"(it has: {','.join(sorted(chain_res))})")
            continue
        absent = [n for n in range(int(lo), int(hi) + 1) if n not in nums[ch]]
        if absent:
            lo_n, hi_n = min(nums[ch]), max(nums[ch])
            problems.append(
                f"contig {ch}{lo}-{hi} names {len(absent)} residue(s) the file does "
                f"not have; chain {ch} runs {lo_n}-{hi_n}")

    for tok in [t.strip() for t in hotspots.split(",") if t.strip()]:
        m = re.fullmatch(r"([A-Za-z])(\d+)", tok)
        if not m:
            problems.append(f"hotspot '{tok}' is not in the form A59")
            continue
        ch, n = m.group(1), int(m.group(2))
        if ch not in chain_res:
            problems.append(f"hotspot {tok}: chain {ch} is not in the file")
        elif n not in nums[ch]:
            problems.append(f"hotspot {tok}: chain {ch} has no residue {n}")
        elif contig_chains and ch not in contig_chains:
            problems.append(
                f"hotspot {tok}: chain {ch} is in the file but the contig never "
                f"names it, so it is not part of the target and this hotspot "
                f"would do nothing (contig uses: {','.join(sorted(contig_chains))})")

if problems:
    for p in problems:
        print("PROBLEM: " + p)
    raise ValueError("The settings do not match the structure - see above. "
                     "Nothing was run.")

print(bar)
print("TARGET CHECK" + (f"  -  {os.path.basename(pdb_path)}" if pdb_path
                        else "  -  no target (monomer)"))
print(bar)
print(f"design_type: {design_type}")
print(f"contigs:     {contigs}")

if pdb_path:
    for c in sorted(chain_res):
        res = chain_res[c]
        ns = sorted(n for n in (resnum(r) for r, _ in res) if n is not None)
        used = c in locals().get("contig_chains", set())
        print(f"chain {c}:  {len(res)} residues, numbered {ns[0]}-{ns[-1]}"
              + ("" if used else "   (not named in the contig)"))
        gaps = [(ns[i], ns[i + 1]) for i in range(len(ns) - 1) if ns[i + 1] - ns[i] > 1]
        if gaps:
            print("   numbering gaps after: "
                  + ", ".join(f"{a} (jumps to {b})" for a, b in gaps[:5]))
        if used:
            seq = "".join(AA3TO1.get(rn, "X") for _, rn in res)
            for i in range(0, len(seq), 50):
                print("   " + str(res[i][0]).rjust(5) + "  "
                      + " ".join(seq[j:j + 10] for j in range(i, min(i + 50, len(seq)), 10)))

    if hotspots.strip():
        print("hotspots:")
        for tok in [t.strip() for t in hotspots.split(",") if t.strip()]:
            ch, n = tok[0], tok[1:]
            rn = next(r[1] for r in chain_res[ch] if r[0] == n)
            print(f"   {tok.ljust(8)}{rn}  {AA3TO1.get(rn, 'X')}")
        arom = sum(1 for t in hotspots.split(",") if t.strip() and
                   AA3TO1.get(next(r[1] for r in chain_res[t.strip()[0]]
                                   if r[0] == t.strip()[1:]), "X") in "FYW")
        print(f"   {len([t for t in hotspots.split(',') if t.strip()])} residues, "
              f"{arom} aromatic (F/Y/W)")
    else:
        print("hotspots:    none - the binder will dock wherever it likes, which")
        print("             is rarely the site you care about")

_designed = re.findall(r"(?<![A-Za-z\d])(\d+)-(\d+)", contigs)
if _designed:
    print("designed:    " + ", ".join(f"{a}-{b} residues" for a, b in _designed))
print(f"run:         {num_designs} design(s), {iterations} diffusion steps")
print(bar)

opts = [
    f"inference.output_prefix=/content/outputs/{name}",
    f"inference.num_designs={num_designs}",
    f"contigmap.contigs=[{contigs}]",
]
opts.append(f"diffuser.partial_T={iterations}" if design_type == "partial diffusion"
            else f"diffuser.T={iterations}")
if pdb_path:
    opts.append(f"inference.input_pdb={pdb_path}")
if hotspots.strip():
    opts.append(f"ppi.hotspot_res=[{hotspots.strip()}]")

_base_yaml = "/content/RFdiffusion/config/inference/base.yaml"
if (FAST or SMOKE) and os.path.isfile(_base_yaml):
    if "write_trajectory" in open(_base_yaml).read():
        opts.append("inference.write_trajectory=False")

cmd = [PY, "run_inference.py"] + opts
print(" ".join(cmd), "\n" + "-" * 70)

proc = subprocess.Popen(
    cmd, cwd="/content/RFdiffusion",
    env=dict(os.environ, DGLBACKEND="pytorch", LD_LIBRARY_PATH=LDPATH),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

_mins = (time.time() - _t0) / 60
print("-" * 70)
print(f"elapsed: {_mins:.1f} min")

if proc.returncode:
    print("FAILED - see the traceback above")
elif CHECK:
    made = sorted(glob.glob(f"/content/outputs/{name}_*.pdb"))
    if not made:
        print("NO FILE - the run reported success but wrote no PDB. Something is wrong")
        print("with the output path, not with your settings.")
    else:
        f = made[0]
        atoms = sum(1 for l in open(f) if l.startswith("ATOM"))
        res = len({l[22:27] for l in open(f) if l.startswith("ATOM")})
        print(f"export OK: {os.path.basename(f)}  "
              f"{os.path.getsize(f) / 1024:.0f} KB, {atoms} atoms, {res} residues")

        out_dir = globals().get("DRIVE_OUTPUT")
        if out_dir and os.path.isdir(os.path.dirname(out_dir)):
            os.makedirs(out_dir, exist_ok=True)
            import shutil
            shutil.copy2(f, out_dir)
            landed = os.path.join(out_dir, os.path.basename(f))
            if os.path.isfile(landed):
                print("Drive copy OK: "
                      + landed.replace("/content/drive/MyDrive", "My Drive"))
            else:
                print("Drive copy FAILED - the run is fine, saving to Drive is not")
        else:
            print("Drive not mounted, so the Drive half was not tested - run step 2.")

        print("\nThe whole chain works. Set run_mode to Production and your real")
        print(f"contigs ({_real_contigs}) will be used; delete the {name}_* files when")
        print("you like, they are 2-step noise.")
elif SMOKE:
    print("Smoke test passed: the environment, the weights and the output path all work.")
    print("Anything that fails now is in your run settings, not the install.")
elif FAST:
    print("Fast debug finished: the settings are accepted and the pipeline runs.")
    print("Set run_mode back to Production for designs worth keeping.")
else:
    print("Done")
    if num_designs > 1:
        print(f"about {_mins / num_designs:.1f} min per design")
!ls -lh /content/outputs/

RuntimeError: Setup has not completed in this runtime. Run steps 3-6 first; step 6 writes the marker this checks. If they did run, one of them failed - scroll up.

In [ ]:
#@title 8. View a design
design = 0 #@param {type:"integer"}
style = "cartoon" #@param ["cartoon", "stick", "sphere"]
color = "spectrum" #@param ["spectrum", "chain", "ssJmol"]

import glob, os
try:
    import py3Dmol
except ImportError:
    !pip install -q py3Dmol
    import py3Dmol

pdbs = sorted(glob.glob(f"/content/outputs/{name}_*.pdb"))
print("found:", [os.path.basename(p) for p in pdbs] or "nothing - did step 7 succeed?")

if pdbs:
    design = min(design, len(pdbs) - 1)
    print("showing:", os.path.basename(pdbs[design]))
    view = py3Dmol.view(width=760, height=520)
    view.addModel(open(pdbs[design]).read(), "pdb")
    key = "colorscheme" if color == "ssJmol" else "color"
    view.setStyle({style: {key: color}})
    view.setBackgroundColor("white")
    view.zoomTo()
    view.show()

In [ ]:
#@title 9. Save results to Drive
#@markdown Colab wipes /content when the runtime ends. Run this before closing the tab.
#@markdown Leave the field blank to use the outputs folder created in step 2.
folder = "" #@param {type:"string"}

import os
from google.colab import drive

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

dest = (f"/content/drive/MyDrive/{folder}" if folder.strip()
        else globals().get("DRIVE_OUTPUT", "/content/drive/MyDrive/RFdiffusion/outputs"))
os.makedirs(dest, exist_ok=True)
!cp -r /content/outputs/* "{dest}/" 2>/dev/null
!ls -lh "{dest}"
print(f"\nSaved to Drive: {dest.replace('/content/drive/MyDrive', 'My Drive')}")


---
## If something fails

**A Run all stopped partway.** That is the environment check in step 4 or the import
check in step 6 doing its job - read the message it raised, which names what failed.
Neither is a transient error worth retrying in place; see the next paragraph.

**Never re-run a setup step on top of a partial failure.** The install is skipped because a
folder already exists, and a later step then fails for a reason unrelated to the real
problem. Instead: Runtime -> **Disconnect and delete runtime**, then start from step 1. A
plain restart keeps the files, so it keeps the problem.

**`libcusparse.so.11: cannot open shared object file`** - DGL cannot find the CUDA 11
libraries. Step 4 prints where `libcusparse.so.11` lives; if that says NOT FOUND, the
`nvidia-*-cu11` installs failed, usually from a full disk.

**`module compiled using NumPy 1.x cannot be run in NumPy 2.x`** - something reinstalled
NumPy 2. Re-run the last pip line of step 4 to pin it back to 1.26.4.

**Step 7 rejects a config key** - flag names shift between RFdiffusion versions. Run
`!cd /content/RFdiffusion && /content/rfenv/bin/python run_inference.py --help`.

**Out of memory** - shorten the contig or lower the iterations. A free T4 manages a few
hundred residues.

**Is it my settings or the install?** Set step 7's `run_mode` to **Smoke test**. It
ignores every field and diffuses a 20-residue monomer in well under a minute. If that
works the environment is fine and the problem is in the run settings; if it fails, the
problem is upstream of them and the steps above apply.

## What comes after this

RFdiffusion produces a backbone - a shape, not yet a protein. Two further steps are needed
before anything is orderable:

1. **ProteinMPNN** designs an amino acid sequence onto the backbone.
2. **AlphaFold** checks that sequence folds back into the intended shape.

Those need their own environments, so they are not bolted on here. The
[ProteinMPNN notebook](https://colab.research.google.com/github/dauparas/ProteinMPNN/blob/main/colab_notebooks/quickdemo.ipynb)
takes these PDB files directly, and
[ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)
validates the sequences it returns.

For binder design the usual filter is AlphaFold predicting the complex confidently with the
binder landing within ~2 A of the design. Expect a low hit rate - that is normal, and it is
why designs are generated in bulk.